# 2025/26-XAI-LABS

Make sure, you have completed these exercises and have discussed them with a teacher. 

Please return a Jupyter notebook as a submission in Canvas, for record keeping.

**Group No:** Project -Assignment 27


**Student Name**: Alexander Diyanov Kralev

**Student Name**: Airidas Radzvilas

# Feature Visualization — Practical Exercises

**Prerequisites:** PyTorch (or TensorFlow), a pretrained CNN such as VGG16 or ResNet50 from `torchvision.models`, and a small set of test images (e.g. a few ImageNet samples or CIFAR-10).

**Suggested libraries:** `torch-lucent`, `lucid`, or a custom optimization loop using `torch.autograd`.

## Exercise 1 — Activation Maximization Across Layers

### Objective
Generate synthetic images that maximally activate specific channels in different layers of a pretrained CNN, then verify the lecture's claim that early layers learn edges, middle layers learn textures/patterns, and late layers learn object parts.

### Task
Pick a pretrained CNN (e.g. VGG16). Select **one channel** from each of three layers: an early conv layer, a middle conv layer, and a late conv layer. For each chosen channel, optimize an input image (starting from random noise) to maximize the mean activation of that channel. Apply standard regularization techniques such as jitter, small rotations, and scaling during optimization to obtain visually meaningful results.


- Display a grid of generated images: one row per layer (early / middle / late), at least 3 channels per row.


In [ ]:
## Your code here. 

## Exercise 2 — Filter and Feature-Map Visualization

### Objective
Distinguish between **filters** (the learned weights) and **feature maps** (the activations produced when an image passes through those filters), and observe how they evolve across the depth of a CNN.

### Task
For a pretrained CNN:

1. Extract and visualize the learned filters of the **first convolutional layer** directly as small RGB images (one image per filter).
2. Pass an input image of your choice through the network and visualize the **feature maps** for several channels in the first, a middle, and the last convolutional layer.
3. Display the original image alongside its feature maps for easy comparison.

### Deliverables
- A figure showing all (or the first 32) filters of layer 1 as a tiled image.
- Three figures showing feature maps from early / middle / late layers for the same input image.

In [ ]:
# your code here

## Exercise 3 — Dataset Examples vs. Optimized Inputs
### Objective
Compare two ways of "seeing" what a unit detects: (a) finding real images that strongly activate it, and (b) optimizing a synthetic image to maximize its activation. This makes concrete the lecture's caution that dataset-based examples may reflect dataset correlations rather than the unit's true focus.
### Task
Choose a single channel in the **last convolutional layer** of a pretrained CNN. Then:
1. Run a subset of a labeled dataset (e.g. 1,000 images from ImageNet validation or CIFAR-10) through the model and record the mean activation of your chosen channel for each image. Display the **top-9 images** that activate it most strongly.
2. Generate one **activation-maximization image** for the same channel (you can reuse code from Exercise 7).
3. Place the top-9 dataset images and the synthetic image side by side.
### Deliverables
- A side-by-side figure: the 3×3 grid of top dataset images and the optimized synthetic image.
- Think about:
  - Do the two visualizations agree on what the channel "detects"?
  - Are there hints of dataset correlations (e.g. the unit always fires on green grass not because it detects grass, but because dogs in the training set were photographed outdoors)?
  - Which method gives you more confidence about the unit's role, and why?

In [ ]:
# your code here

## Compare Pixel Attribution Methods

Apply **Saliency Maps**, **CAM**, **Grad-CAM**, and **SmoothGrad** to a set of images and analyze the quality of the explanations provided by each method. 

You can use any of the following implementations:

- **PyTorch:** [`captum`](https://captum.ai/) (Saliency, SmoothGrad via `NoiseTunnel`, GradCAM via `LayerGradCam`, Integrated Gradients, Guided GradCAM) and [`pytorch-grad-cam`](https://github.com/jacobgil/pytorch-grad-cam) (full family of CAM methods, including GradCAM, GradCAM++, ScoreCAM, EigenCAM).
- **TensorFlow / Keras:** [`tf-keras-vis`](https://github.com/keisen/tf-keras-vis) — includes `Saliency`, `SmoothGrad`, `GradCAM`, `GradCAM++`, and `ScoreCAM` in a single package.
- Or implement the methods from scratch using `torch.autograd` / `tf.GradientTape`.

You are expected to compare the following techniques:

- **Saliency Maps:** Compute the gradient of the class score with respect to the input pixels.
- **CAM:** Compute the class activation map using the weighted sum of feature maps from the last convolutional layer (requires a CNN with a Global Average Pooling layer before the classifier, e.g. ResNet).
- **Grad-CAM:** Compute the gradient-weighted class activation map; works on most CNN architectures without retraining.
- **SmoothGrad:** Average saliency maps computed over multiple noisy versions of the input image.

### Exercise 4

Pick a pretrained CNN (e.g. ResNet50 or VGG16) and a set of at least 3–5 input images of different classes. For each image, compute the saliency map, CAM, Grad-CAM, and SmoothGrad attribution. Choose appropriate target layers (typically the last convolutional layer for CAM and Grad-CAM) and reasonable hyperparameters (e.g. number of noisy samples and noise level for SmoothGrad).

### Exercise 5

Create a function that displays the original image alongside the saliency map, CAM, Grad-CAM, and SmoothGrad visualizations side by side, with a clear title for each panel and a colorbar for the heatmaps. Apply the function to all your chosen images and present the visualizations.

### Exercise 6

Compare the results. Some points to look for are:

- Which technique provides the most intuitive explanation?
- The strengths and weaknesses of each technique (e.g. resolution, noise, class-discriminativeness, architectural constraints).
- Any observed differences in the explanations across different images.
- Cases where the methods disagree — and which one (if any) seems more trustworthy.

In [2]:
# your code here

## Pixel Attribution for Text — Grad-CAM on a Text-CNN

In the image exercises above, you applied Saliency, CAM, and Grad-CAM to 2-D convolutional networks over images. The same algorithms apply to **1-D convolutional networks over text** (Kim, 2014), where convolutions slide over a sequence of word embeddings instead of a 2-D pixel grid. The "spatial" axis is now the **token position**, and the resulting heatmap is a 1-D vector that you can visualize as token-level highlighting on the original sentence.

This works because a text-CNN preserves a position-aligned feature map all the way through the conv layers, exactly like an image CNN — so the lecture's reasoning about *why the last conv layer is the right Grad-CAM target* transfers directly. (For transformer-based text classifiers, this is no longer the case, since self-attention mixes positions and there is no convolutional feature map with a preserved sequential layout. Attention rollout and Integrated Gradients are the analogous methods there — outside the scope of this exercise.)

### Pretrained model

A pretrained text-CNN is provided so you can focus on the explanation methods rather than on training. Download the model and tokenizer from **[LINK TO BE PROVIDED]**. The architecture is a standard Kim (2014)-style network: an embedding layer, parallel 1-D conv layers with kernel sizes 3, 4, and 5, ReLU + global max-pool over the sequence axis, and a final linear classifier. The dataset, label set, and a starter loading snippet are included alongside the weights.

You can use any of the following implementations for the explanation methods:

- **PyTorch:** [`captum`](https://captum.ai/) — `LayerGradCam`, `LayerAttribution.interpolate` (to upsample the 1-D Grad-CAM map back to token resolution), and `Saliency` for the gradient baseline. Captum's GradCAM works on 1-D conv layers without modification.
- **TensorFlow / Keras:** [`tf-keras-vis`](https://github.com/keisen/tf-keras-vis) — `GradCAM` and `Saliency` work on 1-D conv layers; you'll need a small wrapper that maps the resulting 1-D heatmap back to tokens.
- Or implement Grad-CAM from scratch using `torch.autograd` / `tf.GradientTape` — the formulas from the lecture (αᵏᶜ = (1/Z) Σ ∂Sᶜ/∂fᵏ, then ReLU(Σ αᵏᶜ fᵏ)) work unchanged in 1-D.

You are expected to compare the following techniques on text:

- **Vanilla Saliency:** ∂Sᶜ/∂x reduced across the embedding dimension to produce one importance score per token.
- **Grad-CAM:** Computed at the last 1-D convolutional layer for the predicted class, then upsampled back to the token sequence length.
- **SmoothGrad:** Saliency averaged over multiple input perturbations. Since adding Gaussian noise to discrete token IDs is meaningless, perturb the **embeddings** instead.

### Exercise 7

Load the pretrained text-CNN and its tokenizer. Verify the model works correctly by running it on a few example sentences and printing the predicted class and confidence. Report the model's accuracy on the provided test set and produce a confusion matrix. Identify and save:

- at least 2 sentences the model classifies **correctly with high confidence**,
- at least 2 sentences the model classifies **incorrectly**,
- at least 1 sentence the model classifies with **low confidence** (close to the decision boundary).

You will use these in Exercise 10. Also locate the **last 1-D convolutional layer** in the model — the Grad-CAM target — and confirm its output shape is `(batch, channels, sequence_length)` before any pooling.

### Exercise 8

For each of the sentences you saved in Exercise 9, compute:

1. **Vanilla Saliency** over the input tokens. Reduce the embedding-dimension gradient to one score per token (e.g. via L2 norm or sum).
2. **Grad-CAM** at the last 1-D conv layer for the predicted class, then interpolate the 1-D heatmap to the original token sequence length.
3. **SmoothGrad** over the input embeddings — add Gaussian noise to the embeddings, recompute saliency, and average over ~25–50 noisy samples.

Create a function that displays the original sentence with **per-token highlighting**, where each token is colored by its attribution score (e.g. via background color intensity or HTML rendering — `captum.attr.visualization.visualize_text` works well in notebooks). Display the original sentence and the three attribution visualizations stacked vertically for each example, with a clear title for each method.

### Exercise 9

Compare the results. Some points to look for are:

- Which technique provides the most intuitive explanation for text? Do the highlighted tokens align with words a human would expect to drive the prediction (sentiment-bearing words, topic-specific terms, etc.)?
- Strengths and weaknesses of each method on text specifically: token-level resolution, noise level, sensitivity to padding tokens, behavior on out-of-vocabulary words.
- For your misclassified examples, do the explanations help diagnose **why** the model failed?
- If the model has parallel conv branches with different kernel sizes (e.g. 3, 4, 5), compute Grad-CAM separately for each branch. Does the 3-gram branch highlight different tokens than the 5-gram branch? What does that tell you about what each branch has learned?
- Cases where the methods disagree — and which one (if any) seems more trustworthy. Connect this back to the *Can we trust these explanations?* slide from the lecture.

In [ ]:
# your code here

## Pixel Attribution for Text — Grad-CAM on a Text-CNN

In the image exercises above, you applied Saliency, CAM, and Grad-CAM to 2-D convolutional networks over images. The same algorithms apply to **1-D convolutional networks over text** (Kim, 2014), where convolutions slide over a sequence of word embeddings instead of a 2-D pixel grid. The "spatial" axis is now the **token position**, and the resulting heatmap is a 1-D vector that you can visualize as token-level highlighting on the original sentence.

This works because a text-CNN preserves a position-aligned feature map all the way through the conv layers, exactly like an image CNN — so the lecture's reasoning about *why the last conv layer is the right Grad-CAM target* transfers directly. (For transformer-based text classifiers, this is no longer the case, since self-attention mixes positions and there is no convolutional feature map with a preserved sequential layout. Attention rollout and Integrated Gradients are the analogous methods there — outside the scope of this exercise.)

### Pretrained model

A pretrained text-CNN is provided so you can focus on the explanation methods rather than on training. Download the model and tokenizer from **[LINK TO BE PROVIDED]**. The architecture is a standard Kim (2014)-style network: an embedding layer, parallel 1-D conv layers with kernel sizes 3, 4, and 5, ReLU + global max-pool over the sequence axis, and a final linear classifier. The dataset, label set, and a starter loading snippet are included alongside the weights.

You can use any of the following implementations for the explanation methods:

- **PyTorch:** [`captum`](https://captum.ai/) — `LayerGradCam`, `LayerAttribution.interpolate` (to upsample the 1-D Grad-CAM map back to token resolution), and `Saliency` for the gradient baseline. Captum's GradCAM works on 1-D conv layers without modification.
- **TensorFlow / Keras:** [`tf-keras-vis`](https://github.com/keisen/tf-keras-vis) — `GradCAM` and `Saliency` work on 1-D conv layers; you'll need a small wrapper that maps the resulting 1-D heatmap back to tokens.
- Or implement Grad-CAM from scratch using `torch.autograd` / `tf.GradientTape` — the formulas from the lecture (αᵏᶜ = (1/Z) Σ ∂Sᶜ/∂fᵏ, then ReLU(Σ αᵏᶜ fᵏ)) work unchanged in 1-D.

You are expected to compare the following techniques on text:

- **Vanilla Saliency:** ∂Sᶜ/∂x reduced across the embedding dimension to produce one importance score per token.
- **Grad-CAM:** Computed at the last 1-D convolutional layer for the predicted class, then upsampled back to the token sequence length.
- **SmoothGrad:** Saliency averaged over multiple input perturbations. Since adding Gaussian noise to discrete token IDs is meaningless, perturb the **embeddings** instead.

### Exercise 10

Load the pretrained text-CNN and its tokenizer. Verify the model works correctly by running it on a few example sentences and printing the predicted class and confidence. Report the model's accuracy on the provided test set and produce a confusion matrix. Identify and save:

- at least 2 sentences the model classifies **correctly with high confidence**,
- at least 2 sentences the model classifies **incorrectly**,
- at least 1 sentence the model classifies with **low confidence** (close to the decision boundary).

You will use these in Exercise 10. Also locate the **last 1-D convolutional layer** in the model — the Grad-CAM target — and confirm its output shape is `(batch, channels, sequence_length)` before any pooling.

### Exercise 11

For each of the sentences you saved in Exercise 10, compute:

1. **Vanilla Saliency** over the input tokens. Reduce the embedding-dimension gradient to one score per token (e.g. via L2 norm or sum).
2. **Grad-CAM** at the last 1-D conv layer for the predicted class, then interpolate the 1-D heatmap to the original token sequence length.
3. **SmoothGrad** over the input embeddings — add Gaussian noise to the embeddings, recompute saliency, and average over ~25–50 noisy samples.

Create a function that displays the original sentence with **per-token highlighting**, where each token is colored by its attribution score (e.g. via background color intensity or HTML rendering — `captum.attr.visualization.visualize_text` works well in notebooks). Display the original sentence and the three attribution visualizations stacked vertically for each example, with a clear title for each method.

### Exercise 12

Compare the results. Some points to look for are:

- Which technique provides the most intuitive explanation for text? Do the highlighted tokens align with words a human would expect to drive the prediction (sentiment-bearing words, topic-specific terms, etc.)?
- Strengths and weaknesses of each method on text specifically: token-level resolution, noise level, sensitivity to padding tokens, behavior on out-of-vocabulary words.
- For your misclassified examples, do the explanations help diagnose **why** the model failed?
- If the model has parallel conv branches with different kernel sizes (e.g. 3, 4, 5), compute Grad-CAM separately for each branch. Does the 3-gram branch highlight different tokens than the 5-gram branch? What does that tell you about what each branch has learned?
- Cases where the methods disagree — and which one (if any) seems more trustworthy. Connect this back to the *Can we trust these explanations?* slide from the lecture.

In [ ]:
# your code here